# Assignment 2: NVIDIA Nemotron comparison

Run these cells in order with Internet enabled and CPU selected; no Kaggle GPU is needed.
Enable the Kaggle secret NVIDIA_API_KEY for this notebook.

The model is nvidia/nemotron-3.5-lightning-30b-a3b through NVIDIA's hosted API.
The runner evaluates naive, custom and stall_reflection on BabyAI goto, Crafter and MiniHack MazeWalk 9x9,
using seeds 42â€“46, one episode per benchmark/seed, 50 steps, text history 16,
temperature 0 and at most 1,024 output tokens per call: 45 new episodes.
Built-in thinking is disabled for all three agents; the custom planning and reflection prompts remain active.
This setting is recorded in each run manifest.

Completed runs in the result directory are skipped. API errors stop the evaluation and do not count as scores.
Qwen results are stored separately. Download the raw result ZIP after the run for later analysis and the single report.
Free API access may be rate limited; completion in one session is not guaranteed.


In [ ]:
from pathlib import Path
import os
import json
import subprocess
import tempfile

CLIENT = "nvidia"
MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"
PORT = 18080

if CLIENT == "vllm":
    gpus = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
    ).strip().splitlines()
    if len(gpus) != 2:
        raise RuntimeError("Select GPU T4 x2.")
    print("GPUs:", *gpus, sep="\n- ")
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    if CLIENT == "gemini":
        os.environ["GEMINI_API_KEYS"] = json.dumps([
            secrets.get_secret(f"GEMINI_API_KEY_{i}") for i in range(1, 6)
        ])
        os.environ.pop("GEMINI_API_KEY", None)
        os.environ.pop("GOOGLE_API_KEY", None)
    else:
        secret_name = {"openai": "OPENAI_API_KEY", "claude": "ANTHROPIC_API_KEY", "nvidia": "NVIDIA_API_KEY"}[CLIENT]
        os.environ[secret_name] = secrets.get_secret(secret_name)
    if MODEL == "Qwen/Qwen2.5-7B-Instruct":
        raise ValueError("Set MODEL to the exact cloud model ID.")

TEMP_ROOT = Path("/kaggle/temp")
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(TEMP_ROOT / "huggingface")
PROJECT = Path(tempfile.mkdtemp(prefix="balrog-", dir=TEMP_ROOT)) / "agentic-llm-assignment-2"
subprocess.run([
    "git", "clone", "--branch", "siddique/assignment-foundation", "--single-branch",
    "https://github.com/siddiqueMangi2003/agentic-llm-assignment-2.git", str(PROJECT),
], check=True)
RESULTS = Path("/kaggle/working/assignment_results")
print("Repository:", PROJECT)

In [ ]:
import sys

ENV = TEMP_ROOT / "balrog-py310"
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
PYTHON = ENV / "bin" / "python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.10", str(ENV)], check=True)
packages = ["-e", str(PROJECT), "setuptools<81"]
if CLIENT == "vllm":
    packages.append("vllm==0.31.0")
subprocess.run(
    [sys.executable, "-m", "uv", "pip", "install", "--python", str(PYTHON), *packages],
    check=True,
)
os.environ["PATH"] = str(ENV / "bin") + os.pathsep + os.environ["PATH"]

In [ ]:
import time
from urllib.error import URLError
from urllib.request import urlopen

command = [
    str(PYTHON), "scripts/run_assignment.py",
    "--model", MODEL, "--client", CLIENT,
    "--output", str(RESULTS),
]

server = None
log_path = Path("/kaggle/working/vllm.log")
try:
    if CLIENT == "vllm":
        server_cmd = [
            str(ENV / "bin" / "vllm"), "serve", MODEL,
            "--host", "127.0.0.1", "--port", str(PORT),
            "--tensor-parallel-size", "2",
            "--max-model-len", "16384", "--gpu-memory-utilization", "0.85",
        ]
        with log_path.open("w") as log:
            server = subprocess.Popen(server_cmd, stdout=log, stderr=subprocess.STDOUT)
        for _ in range(240):
            if server.poll() is not None:
                raise RuntimeError("vLLM stopped:\n" + log_path.read_text()[-4000:])
            try:
                with urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=2):
                    break
            except (URLError, TimeoutError):
                time.sleep(5)
        else:
            raise TimeoutError("vLLM did not start:\n" + log_path.read_text()[-4000:])
    subprocess.run(command, cwd=PROJECT, check=True)
finally:
    if server is not None:
        server.terminate()
        try:
            server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait()

In [ ]:
import shutil

archive = shutil.make_archive(
    "/kaggle/working/assignment-results", "zip",
    root_dir=RESULTS.parent, base_dir=RESULTS.name,
)
print("Download raw results:", archive)
